In [ ]:
from vectorbt.returns.nb import *

# get_return_nb
计算单期收益率 $\frac{{output\_value - input\_value}}{{input\_value}}$

```python
@njit(cache=True)
def get_return_nb(input_value: float, output_value: float) -> float:

    if input_value == 0:
        if output_value == 0:
            return 0.
        return np.inf * np.sign(output_value)
    return_value = (output_value - input_value) / input_value
    if input_value < 0:
        return_value *= -1
    return return_value
```

In [ ]:
print(get_return_nb(100.0, 110.0))  # 10%上涨
print(get_return_nb(100.0, 90.0))   # 10%下跌
print(get_return_nb(0.0, 10.0))     # 从零开始
print(get_return_nb(-100.0, -90.0)) # 负基数情况

# returns_1d_nb
计算多期收益率
- 第 i 期收益率 = get_return_nb(第i-1期价格, 第i期价格)
- 第 0 期收益率 = get_return_nb(初始价格, 第0期价格)

```python
@njit(cache=True)
def returns_1d_nb(value: tp.Array1d, init_value: float) -> tp.Array1d:

    out = np.empty(value.shape, dtype=np.float64)
    input_value = init_value
    for i in range(out.shape[0]):
        output_value = value[i]
        out[i] = get_return_nb(input_value, output_value)
        input_value = output_value
    return out
```

In [ ]:
prices = np.array([100.0, 110.0, 105.0, 115.0])
returns_1d_nb(prices, 95.0)

# returns_nb
计算多资产多期收益率

```python
@njit(cache=True)
def returns_nb(value: tp.Array2d, init_value: tp.Array1d) -> tp.Array2d:

    out = np.empty(value.shape, dtype=np.float64)
    for col in range(out.shape[1]):
        out[:, col] = returns_1d_nb(value[:, col], init_value[col])
    return out
```

In [ ]:
prices = np.array([[100.0, 200.0],   # t0: 股票A=100, 股票B=200
                    [110.0, 190.0],   # t1: 股票A=110, 股票B=190
                    [105.0, 210.0]])  # t2: 股票A=105, 股票B=210
init_prices = np.array([95.0, 180.0])  # 初始基准价格
returns_nb(prices, init_prices)

# total_return_apply_nb
计算总收益率 $\mathop \Pi \limits_i \left( {1 + {r_i}} \right) - 1$

```python
@njit(cache=True)
def total_return_apply_nb(idxs: tp.Array1d, col: int, returns: tp.Array1d) -> float:
    return np.nanprod(returns + 1) - 1
```

In [ ]:
returns = np.array([0.1, -0.05, 0.08, 0.02])
print(total_return_apply_nb(None, None, returns))
# 计算过程：
# (1+0.1) × (1-0.05) × (1+0.08) × (1+0.02) - 1

# `cum_returns_`

## cum_returns_1d_nb
一维累计收益率计算：
- start_value ≠ 0：累计收益率[i] = (1+r[0]) × (1+r[1]) × ... × (1+r[i]) × start_value
- start_value = 0：累计收益率[i] = (1+r[0]) × (1+r[1]) × ... × (1+r[i]) - 1

```python
@njit(cache=True)
def cum_returns_1d_nb(returns: tp.Array1d, start_value: float) -> tp.Array1d:
    out = np.empty_like(returns, dtype=np.float64)
    cumprod = 1
    for i in range(returns.shape[0]):
        if not np.isnan(returns[i]):
            cumprod *= returns[i] + 1
        out[i] = cumprod
    if start_value == 0.:
        return out - 1.
    return out * start_value
```

In [ ]:
returns = np.array([0.1, -0.05, 0.08, 0.02])
print(cum_returns_1d_nb(returns, 0.0))  # 相对累计收益
print(cum_returns_1d_nb(returns, 100.0))  # 绝对资产价值

## cum_returns_nb
二维累计收益率计算

```python
@njit(cache=True)
def cum_returns_nb(returns: tp.Array2d, start_value: float) -> tp.Array2d:
    out = np.empty_like(returns, dtype=np.float64)
    for col in range(returns.shape[1]):
        out[:, col] = cum_returns_1d_nb(returns[:, col], start_value)
    return out
```

In [ ]:
returns = np.array([[0.1, 0.05],    # t1: 资产A=10%, 资产B=5%
                     [-0.05, 0.02],  # t2: 资产A=-5%, 资产B=2%
                     [0.08, -0.01]]) # t3: 资产A=8%, 资产B=-1%
cum_returns_nb(returns, 100.0)

## cum_returns_final_1d_nb
等价于 `cum_returns_1d_nb` 的最后一个值：
- start_value ≠ 0：(1+r[0]) × (1+r[1]) × ... × (1+r[end]) × start_value
- start_value = 0： (1+r[0]) × (1+r[1]) × ... × (1+r[end]) - 1

```python
@njit(cache=True)
def cum_returns_final_1d_nb(returns: tp.Array1d, start_value: float = 0.) -> float:

    out = np.nanprod(returns + 1.)
    if start_value == 0.:
        return out - 1.
    return out * start_value
```

## cum_returns_final_nb
等价于 `cum_returns_nb` 的最后一行值：

```python
@njit(cache=True)
def cum_returns_final_1d_nb(returns: tp.Array1d, start_value: float = 0.) -> float:

    out = np.nanprod(returns + 1.)
    if start_value == 0.:
        return out - 1.
    return out * start_value
```

In [ ]:
returns = np.array([[0.1, 0.05],
                     [-0.05, 0.02],
                     [0.08, -0.01]])
print(cum_returns_final_nb(returns, 0.0))

# rolling_cum_returns_final_nb
计算 `returns` 每列滑动窗口长度为 `window` 的累计收益率

参数
- `returns` (tp.Array2d): 二维收益率矩阵，形状为(时间点数, 资产数)
- `window` (int): 滚动窗口大小（观测期数）
- `minp` (tp.Optional[int]): 最小有效观测期数
  - 当窗口内有效数据少于此值时返回NaN
  - None 时使用 window 作为最小期数
- `start_value` (float): 起始价值，默认为 0（相对收益率）

返回：tp.Array2d，滚动累计收益率矩阵，形状与输入相同

```python
@njit
def rolling_cum_returns_final_nb(returns: tp.Array2d,
                                 window: int,
                                 minp: tp.Optional[int],
                                 start_value: float = 0.) -> tp.Array2d:
    def _apply_func_nb(i, col, _returns, _start_value):
        return cum_returns_final_1d_nb(_returns, _start_value)

    return generic_nb.rolling_apply_nb(
        returns, window, minp, _apply_func_nb, start_value)
```

In [ ]:
returns = np.array([[0.02, 0.01],
                     [0.03, -0.01],
                     [-0.01, 0.02],
                     [0.01, 0.01]])
print(rolling_cum_returns_final_nb(returns, window=3, minp=2, start_value=0.0))

# `annualized_return_`

## annualized_return_1d_nb
根据日频/周频/月频收益率序列 `returns` 计算年化收益率

参数
- `returns` (tp.Array1d)：一维收益率时间序列，可以是日频/周频/月频
- `ann_factor` (float)：年化因子
  - 日频：通常为252（年均交易日）
  - 周频：通常为52（年均周数）
  - 月频：通常为12（年均月数）

原理：$年化收益率 = {\left( {1 + returns累计收益率} \right)^{\frac{{{\rm{ann}}\_factor}}{{len(returns)}}}} - 1$

返回：float，年化收益率（小数形式，如 0.15 表示 15%）

```python
@njit(cache=True)
def annualized_return_1d_nb(returns: tp.Array1d, ann_factor: float) -> float:
    end_value = cum_returns_final_1d_nb(returns, 1.)
    return end_value ** (ann_factor / returns.shape[0]) - 1
```

In [ ]:
returns = np.array([0.02, 0.01, -0.005, 0.03])  # 4个月收益率
print(annualized_return_1d_nb(returns, 12.0))  # 月频数据年化
# 计算过程：
# 1. 累计收益率 = (1.02) × (1.01) × (0.995) × (1.03) - 1
# 2. 年化收益率 = (1 + 累计收益率)^(12/4) - 1

## annualized_return_nb
二维版本的 `annualized_return_1d_nb`

```python
@njit(cache=True)
def annualized_return_nb(returns: tp.Array2d, ann_factor: float) -> tp.Array1d:
    out = np.empty(returns.shape[1], dtype=np.float64)
    for col in range(returns.shape[1]):
        out[col] = annualized_return_1d_nb(returns[:, col], ann_factor)
    return out
```

In [ ]:
returns = np.array([[0.01, 0.02, -0.005],   # 资产A, B, C的月收益率
                     [0.02, -0.01, 0.01],
                     [-0.005, 0.015, 0.02],
                     [0.03, 0.01, -0.01]])
print(annualized_return_nb(returns, 12.0))  # 各资产的年化收益率

# rolling_annualized_return_nb

# annualized_volatility_1d_nb

# annualized_volatility_nb

# rolling_annualized_volatility_nb

# drawdown_1d_nb

# drawdown_nb

# max_drawdown_1d_nb

# max_drawdown_nb

# rolling_max_drawdown_nb

# calmar_ratio_1d_nb

# calmar_ratio_nb

# rolling_calmar_ratio_nb

# omega_ratio_1d_nb

# omega_ratio_nb

# rolling_omega_ratio_nb

# sharpe_ratio_1d_nb

# sharpe_ratio_nb

# rolling_sharpe_ratio_nb

# downside_risk_1d_nb

# downside_risk_nb

# rolling_downside_risk_nb

# sortino_ratio_1d_nb

# sortino_ratio_nb

# rolling_sortino_ratio_nb

# information_ratio_1d_nb

# information_ratio_nb

# rolling_information_ratio_nb

# beta_1d_nb

# beta_nb

# rolling_beta_nb

# alpha_1d_nb

# alpha_nb

# rolling_alpha_nb

# tail_ratio_1d_nb

# tail_ratio_nb

# rolling_tail_ratio_nb

# value_at_risk_1d_nb

# value_at_risk_nb

# rolling_value_at_risk_nb

# cond_value_at_risk_1d_nb

# cond_value_at_risk_nb

# rolling_cond_value_at_risk_nb

# capture_1d_nb

# capture_nb

# rolling_capture_nb

# up_capture_1d_nb

# up_capture_nb

# rolling_up_capture_nb

# down_capture_1d_nb

# down_capture_nb

# rolling_down_capture_nb